# Reporte del dataset

Inventario de lo que hay en `data/cleaned/` y en el caché de `data/processed/`: archivos que
entran, anotaciones totales, reparto por clase y por split, y una huella (`sha256`) de la lista
de grabaciones de cada split para comparar dos copias del dataset de un vistazo.

Ejecutar desde la raíz del repo o desde `notebooks/`.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from core.config import CLEANED_DIR, PROCESSED_DIR, RAW_DIR  # noqa: E402
from data import cache  # noqa: E402
from data.annotations import load_annotations  # noqa: E402
from prepare_data import select_experiment  # noqa: E402

pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)
print(ROOT)

## 1. `data/cleaned/` — anotaciones normalizadas

Una `.txt` por grabación con `.wav` al lado en `data/raw/`; es lo que lee todo lo demás.

In [ ]:
annotations = load_annotations()
annotations["recording"] = [
    str(Path(p).relative_to(RAW_DIR)) for p in annotations["audio_path"]
]
annotations["folder"] = annotations["recording"].str.split("/").str[0]

cleaned_files = sorted(p.relative_to(CLEANED_DIR) for p in CLEANED_DIR.rglob("*.txt"))
print(f"archivos .txt en cleaned/ : {len(cleaned_files)}")
print(f"grabaciones con .wav      : {annotations['recording'].nunique()}")
print(f"anotaciones totales       : {len(annotations)}")
print(f"especies                  : {annotations['species'].nunique()}")
print(f"pares species/call_type   : {annotations.groupby(['species', 'call_type']).ngroups}")

In [ ]:
por_carpeta = (
    annotations.groupby("folder")
    .agg(
        grabaciones=("recording", "nunique"),
        anotaciones=("recording", "size"),
        tipos_llamada=("call_type", "nunique"),
        dur_media_s=("duration_s", "mean"),
        bw_medio_hz=("bandwidth_hz", "mean"),
    )
    .round(2)
)
por_carpeta.loc["TOTAL"] = [
    annotations["recording"].nunique(),
    len(annotations),
    annotations.groupby(["species", "call_type"]).ngroups,
    round(annotations["duration_s"].mean(), 2),
    round(annotations["bandwidth_hz"].mean(), 2),
]
por_carpeta

In [ ]:
# Todos los pares species/call_type de cleaned/, incluidos los que el experimento descarta.
pares_cleaned = (
    annotations.groupby(["species", "call_type"])
    .agg(anotaciones=("recording", "size"), grabaciones=("recording", "nunique"))
    .sort_values("anotaciones", ascending=False)
)
print(f"{len(pares_cleaned)} pares | {int(pares_cleaned['anotaciones'].sum())} anotaciones")
pares_cleaned

## 2. El experimento: qué pares sobreviven al filtro

`select_experiment()` es la misma función que usa `prepare_data.py`: aplica `EXCLUDED_PAIRS`,
`JOINED_PAIRS` y `MIN_PAIR_COUNT`. Si esto no coincide entre dos copias, el caché tampoco.

In [ ]:
experiment_df, labels = select_experiment(annotations)
experiment_df["recording"] = [
    str(Path(p).relative_to(RAW_DIR)) for p in experiment_df["audio_path"]
]

print(f"anotaciones del experimento : {len(experiment_df)}")
print(f"grabaciones                 : {experiment_df['recording'].nunique()}")
print(f"clases                      : {len(labels)}")
print(", ".join(labels.names))

## 3. `data/processed/` — metadatos del caché

In [ ]:
meta = cache.meta()
labels_json = cache.read_json("labels.json")
print(json.dumps(meta, indent=2, ensure_ascii=False))
print()
print(f"{len(labels_json)} clases en labels.json")

In [ ]:
archivos_cache = sorted(PROCESSED_DIR.iterdir())
pd.DataFrame(
    [
        {
            "archivo": p.name,
            "MB": round(p.stat().st_size / 1024**2, 2),
            "sha256_16": hashlib.sha256(p.read_bytes()).hexdigest()[:16] if p.stat().st_size < 5e7 else "",
        }
        for p in archivos_cache
    ]
)

## 4. Splits: ventanas, grabaciones y cajas

`*.pt` se abre con `mmap=True`: se leen las cajas y las etiquetas sin traer los mel a memoria
(los tres juntos pesan más de 6 GB).

In [ ]:
def read_split(name: str) -> dict:
    return torch.load(cache.split_path(name), map_location="cpu", mmap=True, weights_only=True)


splits = {name: read_split(name) for name in cache.SPLITS}
sources = {
    name: cache.sources(name, len(data["images"])) for name, data in splits.items()
}
recordings = {
    name: sorted(str(Path(p).relative_to(RAW_DIR)) for p in src.recordings)
    for name, src in sources.items()
}
{name: len(rs) for name, rs in recordings.items()}

In [ ]:
filas = []
for name, data in splits.items():
    boxes = data["boxes"]
    n_cajas = [len(b) for b in boxes]
    filas.append(
        {
            "split": name,
            "ventanas": len(boxes),
            "grabaciones": len(recordings[name]),
            "cajas": int(sum(n_cajas)),
            "ventanas_vacias": sum(1 for n in n_cajas if n == 0),
            "cajas_por_ventana": round(sum(n_cajas) / len(n_cajas), 3),
            "max_cajas": max(n_cajas),
            "mel": tuple(data["images"].shape[1:]),
            "dtype": str(data["images"].dtype),
            "GB": round(cache.split_path(name).stat().st_size / 1024**3, 3),
        }
    )

resumen = pd.DataFrame(filas).set_index("split")
resumen.loc["TOTAL"] = [
    resumen["ventanas"].sum(),
    resumen["grabaciones"].sum(),
    resumen["cajas"].sum(),
    resumen["ventanas_vacias"].sum(),
    round(resumen["cajas"].sum() / resumen["ventanas"].sum(), 3),
    resumen["max_cajas"].max(),
    resumen["mel"].iloc[0],
    resumen["dtype"].iloc[0],
    round(resumen["GB"].sum(), 3),
]
resumen

In [ ]:
# Reparto en % sobre el total (SPLIT_RATIOS es 0.6 / 0.225 / 0.175 por archivo de audio).
proporciones = resumen.drop(index="TOTAL")[["ventanas", "grabaciones", "cajas"]]
(100 * proporciones / proporciones.sum()).round(2)

### Ninguna grabación puede caer en dos splits

El split es por archivo de audio; si esto falla hay fuga entre train y test.

In [ ]:
solapes = {
    f"{a}&{b}": sorted(set(recordings[a]) & set(recordings[b]))
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]
}
for par, comunes in solapes.items():
    print(f"{par}: {len(comunes)} grabaciones en común")

todas = sorted(set().union(*recordings.values()))
print(f"\ngrabaciones en el caché      : {len(todas)}")
print(f"grabaciones del experimento  : {experiment_df['recording'].nunique()}")
faltan = sorted(set(experiment_df["recording"]) - set(todas))
print(f"en el experimento y no en el caché: {len(faltan)}")
for r in faltan:
    print(f"  {r}")

### Cajas por clase y split

`cajas` cuenta apariciones en ventanas: una anotación larga cae en varias ventanas
(`clip_len_s` 3 s con `clip_hop_s` 1.5 s), así que siempre supera a `anotaciones`, que son las
anotaciones únicas de las grabaciones asignadas a ese split.

In [ ]:
split_de_grabacion = {r: name for name, rs in recordings.items() for r in rs}
experiment_df["split"] = experiment_df["recording"].map(split_de_grabacion)

cajas = pd.DataFrame(
    {
        name: pd.Series(torch.cat(data["labels"]).numpy()).value_counts()
        for name, data in splits.items()
    }
).fillna(0).astype(int)
cajas.index = [labels.names[i] for i in cajas.index]

unicas = (
    experiment_df.pivot_table(index="label", columns="split", values="recording", aggfunc="size")
    .reindex(columns=list(cache.SPLITS))
    .fillna(0)
    .astype(int)
)

por_clase = pd.concat(
    {"cajas": cajas.reindex(index=unicas.index)[list(cache.SPLITS)], "anotaciones": unicas}, axis=1
)
por_clase[("cajas", "total")] = cajas.sum(axis=1)
por_clase[("anotaciones", "total")] = unicas.sum(axis=1)
por_clase.loc["TOTAL"] = por_clase.sum()
por_clase.sort_values(("anotaciones", "total"), ascending=False)

In [ ]:
# Anotaciones únicas por grabación y split, por si hay que rastrear una clase rara.
grabaciones_por_clase = (
    experiment_df.pivot_table(index="label", columns="split", values="recording", aggfunc="nunique")
    .reindex(columns=list(cache.SPLITS))
    .fillna(0)
    .astype(int)
)
grabaciones_por_clase["total"] = grabaciones_por_clase.sum(axis=1)
grabaciones_por_clase.sort_values("total", ascending=False)

## 5. Huella para comparar dos copias

`recordings_sha256` resume la lista ordenada de grabaciones de cada split. Si dos copias del
dataset dan la misma huella en los tres splits, el reparto es idéntico; si difiere, las tablas
de arriba dicen dónde.

In [ ]:
def huella(items) -> str:
    return hashlib.sha256("\n".join(items).encode()).hexdigest()


huellas = pd.DataFrame(
    [
        {
            "split": name,
            "grabaciones": len(recordings[name]),
            "ventanas": len(splits[name]["boxes"]),
            "cajas": int(sum(len(b) for b in splits[name]["boxes"])),
            "recordings_sha256": huella(recordings[name])[:32],
        }
        for name in cache.SPLITS
    ]
).set_index("split")
huellas

In [ ]:
reporte = {
    "cleaned": {
        "archivos_txt": len(cleaned_files),
        "grabaciones": int(annotations["recording"].nunique()),
        "anotaciones": int(len(annotations)),
        "pares": int(annotations.groupby(["species", "call_type"]).ngroups),
        "por_carpeta": {
            k: int(v) for k, v in annotations.groupby("folder")["recording"].size().items()
        },
    },
    "experimento": {
        "anotaciones": int(len(experiment_df)),
        "grabaciones": int(experiment_df["recording"].nunique()),
        "clases": list(labels.names),
    },
    "meta": meta,
    "splits": {
        name: {
            "ventanas": len(splits[name]["boxes"]),
            "grabaciones": len(recordings[name]),
            "cajas": int(sum(len(b) for b in splits[name]["boxes"])),
            "ventanas_vacias": int(sum(1 for b in splits[name]["boxes"] if len(b) == 0)),
            "recordings_sha256": huella(recordings[name]),
            "recordings": recordings[name],
        }
        for name in cache.SPLITS
    },
}

destino = ROOT / "notebooks" / "dataset_report.json"
destino.write_text(json.dumps(reporte, indent=2, ensure_ascii=False))
print(f"{destino} ({destino.stat().st_size / 1024:.1f} KB)")
print("Comparalo con la otra copia:  diff <(jq -S . a.json) <(jq -S . b.json)")